In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, sys

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
# Load Data
df = pd.read_csv(r"../data/GamingStudy_data_clean.csv")
print(f"Shape: {df.shape}")
df.head(3)

Shape: (10772, 60)


,SWL1,SWL2,SWL3,SWL4,SWL5,Hours,streams,SPIN1,SPIN2,SPIN3,...,Work_Unknown,Degree_High school diploma (or equivalent),Degree_Master (or equivalent),"Degree_Ph.D., Psy. D., MD (or equivalent)",Degree_Unknown,Playstyle_Offline Multiplayer,Playstyle_Other,Playstyle_Real Life Friends,Playstyle_Singleplayer,Playstyle_Strangers
0,7,7,7,5,7,25.0,2.0,1.0,0.0,0.0,...,0,0,0,0,0,0,0,1,0,0
1,6,6,6,3,5,5.0,5.0,2.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,1
2,5,2,6,7,6,9.0,2.0,2.0,2.0,2.0,...,0,1,0,0,0,0,0,1,0,0


In [3]:
# Drop nulls from data
df = df.dropna()
df.shape

(10671, 60)

In [4]:
# Create binary target variable based on GAD_T score
THRESHOLD = 10
y = (df["GAD_T"] >= THRESHOLD).astype(int)

counts = y.value_counts().sort_index()
print(counts)
print(f"  Imbalance ratio : 1 : {counts[0]/counts[1]:.1f}")


GAD_T
0    8842
1    1829
Name: count, dtype: int64
  Imbalance ratio : 1 : 4.8


In [5]:
# Define feature groups
SPIN_FEATURES = [f"SPIN{i}" for i in range(1, 18)]  
SWL_FEATURES  = [f"SWL{i}"  for i in range(1, 6)]

GAME_FEATURES = [
    "Hours", "streams",
    "Platform_PC", "Platform_Smartphone / Tablet",
    "earnings_Other", "earnings_Professional", "earnings_Side Income",
    "whyplay_Improving", "whyplay_Other", "whyplay_Relaxing", "whyplay_Winning",
    "League_Challenger", "League_Diamond", "League_Gold", "League_Master",
    "League_Other", "League_Platinum", "League_Silver", "League_Unknown", "League_Unranked",
    "Playstyle_Offline Multiplayer", "Playstyle_Other",
    "Playstyle_Real Life Friends", "Playstyle_Singleplayer", "Playstyle_Strangers",
]

DEMO_FEATURES = [
    "Age", "Gender_Male", "Gender_Other",
    "Work_Student at college / university", "Work_Student at school",
    "Work_Unemployed / between jobs", "Work_Unknown",
    "Degree_High school diploma (or equivalent)",
    "Degree_Master\xa0(or equivalent)",
    "Degree_Ph.D., Psy. D., MD (or equivalent)", "Degree_Unknown",
]

PERSONALITY_FEATURES = ["Narcissism"]

FEATURES = {
    "A_psych":  SPIN_FEATURES + SWL_FEATURES  + DEMO_FEATURES,
    "B_gaming": GAME_FEATURES + DEMO_FEATURES,
    "C_full":   SPIN_FEATURES + SWL_FEATURES  + GAME_FEATURES
                + DEMO_FEATURES + PERSONALITY_FEATURES,
}


In [6]:
X = df[list(set(SPIN_FEATURES + SWL_FEATURES + GAME_FEATURES + DEMO_FEATURES + PERSONALITY_FEATURES))]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

In [7]:
# Save to pickle
data = {
    "X_train":      X_train,
    "X_test":       X_test,
    "y_train":      y_train,
    "y_test":       y_test,
    "FEATURES":     FEATURES,
}

with open("../data/data.pkl", "wb") as f:
    pickle.dump(data, f)